In [2]:
import sys
import os
import json
from pathlib import Path

# 1. Path Setup
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_rq1 import *
print(f"✓ Success! Project root set to: {_PROJECT_ROOT}")

# 2. Define File Paths
# Point this directly to your test file
TEST_DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "wikidata_testset_RQ1.json") 
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "sparqlce", "models", "wikidata_model_mrr_0.3238.pt") 

TOPKS = [3, 5, 10, 20]
BATCH = 32

# 3. Load the test dataset directly
test_data = load_triplets(TEST_DATA_PATH)

print(f"Dataset Loaded -> Test Split: {len(test_data)}")

# 4. Build corpus and eval items using ONLY the test data
global_corpus, seen_q = build_global_corpus(test_data)
eval_items = build_eval_items(test_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Evaluation Ready -> Anchors: {len(eval_items)} | Corpus: {len(global_corpus)}")

✓ Success! Project root set to: /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings
Dataset Loaded -> Test Split: 186
Evaluation Ready -> Anchors: 186 | Corpus: 2229


In [3]:
# SPARQLCE
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

# TF-IDF
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# E5
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')

# Qwen
evaluate_sentence_transformer('Qwen/Qwen2.5-0.5B-Instruct', 'Qwen2.5-0.5B', eval_items, global_corpus, relevance_scores_per_item, TOPKS)

Loading model from /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/sparqlce/models/wikidata_model_mrr_0.3238.pt...

SPARQLCE
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1649  0.0383  0.0589  0.1998  0.3172    0.1649  0.1522
@5     0.1903  0.0787  0.1023  0.2315  0.4570    0.1903  0.1746
@10    0.1930  0.1559  0.1572  0.2433  0.5430    0.1930  0.1941
@20    0.1890  0.2937  0.2106  0.2509  0.6452    0.1890  0.2451

Baseline: TF-IDF
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0717  0.0143  0.0227  0.0726  0.0914    0.0717  0.0690
@5     0.0796  0.0263  0.0376  0.0804  0.1290    0.0796  0.0754
@10    0.0683  0.0453  0.0516  0.0847  0.1613    0.0683  0.0722
@20    0.0417  0.0621  0.0460  0.0880  0.2097    0.0417  0.061

No sentence-transformers model found with name Qwen/Qwen2.5-0.5B-Instruct. Creating a new one with mean pooling.



E5-base-v2
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0878  0.0177  0.0280  0.1039  0.1290    0.0878  0.0873
@5     0.0968  0.0317  0.0454  0.1139  0.1720    0.0968  0.0939
@10    0.0887  0.0595  0.0666  0.1213  0.2204    0.0887  0.0938
@20    0.0573  0.0822  0.0630  0.1239  0.2581    0.0573  0.0824

Qwen2.5-0.5B
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0950  0.0196  0.0306  0.1013  0.1183    0.0950  0.0930
@5     0.0978  0.0354  0.0484  0.1061  0.1398    0.0978  0.0961
@10    0.0726  0.0506  0.0558  0.1134  0.1935    0.0726  0.0833
@20    0.0419  0.0581  0.0455  0.1164  0.2366    0.0419  0.0667
